# Content Control Classification

In [1]:
# Import packages
import polars as pl
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [2]:
# Read the files
df_top_1000 = pl.read_csv("./data/top_1000_tags.csv")
df_content_control = pl.read_csv("./data/df_unique.csv")

In [3]:
# Turn the series into a list
content_control_list = df_content_control.to_series().to_list()
type(content_control_list)

list

## 1. Test the Classifiers

### 1.1 Test the Gradient Boosting Classifier's Porn Classification

In [4]:
X_porn = df_top_1000["Tag"]
y_porn = df_top_1000["Porn"]

# Split into train and test data
X_train_porn, X_test_porn, y_train_porn, y_test_porn = train_test_split(X_porn, y_porn, test_size=0.2, random_state=42)

# Create a Pipeline
pipeGB = Pipeline([
    ('tfidf', TfidfVectorizer()),('clf', GradientBoostingClassifier())
])

pipeGB.fit(X_train_porn, y_train_porn) # Fit the model
y_predGB = pipeGB.predict(X_test_porn) # Predict the annotation labels
print(f"""Gradient Boosting:
Accuracy: {accuracy_score(y_test_porn, y_predGB):.2f}
Macro F1: {f1_score(y_test_porn, y_predGB, average="macro"):.2f}""")

# Test the model
test_porn_tag = "Avalance"
test_porn_result = pipeGB.predict([test_porn_tag])
print("Test Result:", test_porn_result[0])

Gradient Boosting:
Accuracy: 0.94
Macro F1: 0.87
Test Result: Not Porn


### 1.2 Test the Decision Trees Classifier's Kink Classification

In [5]:
X_kink = df_top_1000["Tag"]
y_kink = df_top_1000["Kink"]

# Split into train and test data
X_train_kink, X_test_kink, y_train_kink, y_test_kink = train_test_split(X_kink, y_kink, test_size=0.2, random_state=42)

# Create a Pipeline
pipeDT = Pipeline([
    ('tfidf', TfidfVectorizer()),('clf', DecisionTreeClassifier())
])

pipeDT.fit(X_train_kink, y_train_kink) # Fit the model
y_predDT = pipeDT.predict(X_test_kink) # Predict the annotation labels
print(f"""Decision Trees:
Accuracy: {accuracy_score(y_test_kink, y_predDT):.2f}
Macro F1: {f1_score(y_test_kink, y_predDT, average="macro"):.2f}""")

# Test the model
test_kink_tag = "Avalance"
test_kink_result = pipeDT.predict([test_kink_tag])
print("Test Result:", test_kink_result[0])

Decision Trees:
Accuracy: 0.94
Macro F1: 0.78
Test Result: Not Kink


## 2. Annotate the Additional Tags's Content-Control Category

In [6]:
def content_control(tag):
    """
    Labels the additional tags.

    Parameter:
        tag (string): The additional tag that should be assigned an annotation label.
    Returns:
        df_temp_content_control (DataFrame): Contains the tag and the Porn and Kink annotation labels
    """
    tag_porn = pipeGB.predict([tag])
    tag_kink = pipeDT.predict([tag])

    df_temp_content_control = pl.DataFrame({
        "Tag": tag,
        "Porn": tag_porn,
        "Kink": tag_kink
    })

    return df_temp_content_control

In [7]:
df_content_control_labels = pl.DataFrame()
# Iterate over the additional tags in the content_control_list
for tag in content_control_list:
    df_content_control_labels = pl.concat([df_content_control_labels, content_control(tag)]) # Concat the results to the df_content_control_labels DataFrame

df_content_control_labels

Tag,Porn,Kink
str,str,str
"""Fluff""","""Not Porn""","""Not Kink"""
"""Angst""","""Not Porn""","""Not Kink"""
"""AvaLance""","""Not Porn""","""Not Kink"""
"""Hurt/Comfort""","""Not Porn""","""Not Kink"""
"""Angst with a Happy Ending""","""Not Porn""","""Not Kink"""
…,…,…
"""Miles Morales is smitten""","""Not Porn""","""Not Kink"""
"""y/n cries""","""Not Porn""","""Not Kink"""
"""texting while fighting""","""Not Porn""","""Not Kink"""


In [8]:
df_content_control_labels.write_csv('data/df_content_control.csv') # Save the result as a CSV file